# Q-Learning 与 Dyna‑Q 实战教程（整合“时序差分算法”与“Dyna‑Q算法”）

> 目标：带你从 **时序差分（TD）** 出发，推导并手写 **Q‑learning**，再扩展到 **Dyna‑Q**（带规划的Q‑learning）。
>
> 你将得到：
> - 一份可运行的、纯 Python 的教学 Notebook（无需额外依赖）。
> - 一个自带 **CliffWalking** 样例环境（Sutton & Barto 书中经典示例的简化实现）。
> - Q‑learning 与 Dyna‑Q 的完整实现、训练与可视化对比。

**参考阅读（中文）**：  
- “时序差分算法”：https://hrl.boyuai.com/chapter/1/时序差分算法  
- “Dyna‑Q算法”：https://hrl.boyuai.com/chapter/1/dyna-q算法

*生成时间：2025-08-27 15:48:04*

## 目录
1. 强化学习与Q函数回顾
2. 时序差分（TD）要点
3. 从TD到 **Q‑learning**（无模型）
4. 代码：自定义 **CliffWalking** 环境（可直接运行）
5. 代码：**Q‑learning** 的实现与训练
6. **Dyna‑Q**：把“学习到的模型”用于规划
7. 代码：**Dyna‑Q** 实现与对比
8. 可视化与策略查看
9. 调参与常见坑
10. 参考与延伸

## 1. 强化学习与 Q 函数回顾

- 我们考虑离散型 **MDP**：状态集合 $\mathcal{S}$、动作集合 $\mathcal{A}$、转移 $P(s'|s,a)$、即时回报 $r$、折扣因子 $\gamma\in(0,1]$。
- **动作价值函数（Q函数）**：
  $$
  Q^\*(s,a) \triangleq \max_\pi \mathbb{E}\Big[\sum_{t=0}^\infty \gamma^t r_{t} \,\big|\, s_0=s,a_0=a,\pi\Big].
  $$
- **贝尔曼最优性方程**：
  $$
  Q^\*(s,a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^\*(s',a') \mid s,a\right].
  $$
- 目标：学习一个近似 $Q(s,a)$，使其满足上式。

## 2. 时序差分（TD）要点

- **TD 思想**：不用等待一个完整回合（蒙特卡罗），而是用**一步预测**来更新估计。
- **TD 误差（Q版本）**：
  $$
  \delta = r + \gamma \max_{a'} Q(s',a') - Q(s,a).
  $$
- 用 $\delta$ 做**自举（bootstrapping）**更新：
  $$
  Q(s,a) \leftarrow Q(s,a) + \alpha \,\delta.
  $$
- 这正是 **Q‑learning** 的核心。

## 3. 从 TD 到 Q‑learning（无模型）

- 交互产生 $(s,a,r,s')$ 样本；
- 采用 $\epsilon$‑greedy 策略（大多选择 $\arg\max_a Q(s,a)$，小概率随机探索）；
- 用下一状态的 **最大动作价值** 作为目标：$r+\gamma\max_{a'}Q(s',a')$；
- 用学习率 $\alpha$ 做一步更新。

**伪代码**：

```
Initialize Q(s,a) arbitrarily
for episode = 1..N:
    s ← env.reset()
    repeat:
        with prob ε: a ← random action
        else:       a ← argmax_a Q(s,a)
        s', r, done ← env.step(a)

        # Q-learning 更新（off-policy, bootstrapping）
        Q(s,a) ← Q(s,a) + α [r + γ max_{a'} Q(s',a') − Q(s,a)]

        s ← s'
    until done
```

## 4. 代码：自定义 CliffWalking 环境

- 4×12 网格；起点 **S** 在左下角，终点 **G** 在右下角；
- 底部 S 与 G 之间为“悬崖 **X**”：一旦踩到得到 −100 回报，并被送回 S（回合**不中止**）；
- 其他每一步回报 −1；到达 **G** 回合结束。

动作编码：`0=右, 1=左, 2=上, 3=下`。

In [ ]:
import numpy as np

class CliffWalkingEnv:
    '''
    一个最小可运行的 CliffWalking 环境（确定性）。
    网格默认 4x12：
    - S: (3,0), G: (3,11)
    - 悬崖 X: (3,1..10)
    - 每步 -1，踩 X 得 -100 并回到 S（回合不中止），到达 G 终止。
    '''
    def __init__(self, height=4, width=12):
        self.height = height
        self.width = width
        self.start = (height - 1, 0)
        self.goal = (height - 1, width - 1)
        self.cliff = {(height - 1, c) for c in range(1, width - 1)}
        self.nA = 4
        self.nS = height * width
        self.pos = None

    def reset(self, seed: int | None = None):
        if seed is not None:
            np.random.default_rng(seed)  # no-op; kept for API compatibility
        self.pos = self.start
        return self._state()

    def _state(self):
        r, c = self.pos
        return r * self.width + c

    def step(self, a: int):
        r, c = self.pos
        if a == 0:   nr, nc = r, c + 1   # 右
        elif a == 1: nr, nc = r, c - 1   # 左
        elif a == 2: nr, nc = r - 1, c   # 上
        elif a == 3: nr, nc = r + 1, c   # 下
        else: raise ValueError("非法动作")

        nr = max(0, min(self.height - 1, nr))
        nc = max(0, min(self.width - 1, nc))
        self.pos = (nr, nc)

        reward = -1.0
        done = False

        if self.pos in self.cliff:
            reward = -100.0
            self.pos = self.start  # 回到起点，回合不中止
        elif self.pos == self.goal:
            done = True

        return self._state(), reward, done, {}

    def render_policy(self, Q: np.ndarray):
        arrows = {0: "→", 1: "←", 2: "↑", 3: "↓"}
        lines = []
        for r in range(self.height):
            row = []
            for c in range(self.width):
                if (r, c) == self.goal:
                    row.append("G")
                elif (r, c) == self.start:
                    row.append("S")
                elif (r, c) in self.cliff:
                    row.append("X")
                else:
                    s = r * self.width + c
                    a = int(np.argmax(Q[s]))
                    row.append(arrows[a])
            lines.append(" ".join(row))
        print("\n".join(lines))

## 5. 代码：Q‑learning 的实现与训练
下面实现标准 **Tabular Q‑learning**，并记录每回合的回报与步数便于可视化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def epsilon_greedy(Q: np.ndarray, s: int, epsilon: float, rng: np.random.Generator, nA: int):
    if rng.random() < epsilon:
        return int(rng.integers(nA))
    return int(np.argmax(Q[s]))

def q_learning(env, episodes=500, alpha=0.5, gamma=0.95,
               epsilon=0.1, epsilon_decay=None, min_epsilon=0.01, seed=0,
               max_steps_per_episode=1000):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.nS, env.nA))
    returns, steps = [], []

    for ep in range(episodes):
        s = env.reset()
        done = False
        G, t = 0.0, 0

        while not done and t < max_steps_per_episode:
            a = epsilon_greedy(Q, s, epsilon, rng, env.nA)
            s2, r, done, _ = env.step(a)

            td_target = r + (0.0 if done else gamma * np.max(Q[s2]))
            Q[s, a] += alpha * (td_target - Q[s, a])

            s = s2
            G += r
            t += 1

        returns.append(G)
        steps.append(t)

        if epsilon_decay is not None:
            epsilon = max(min_epsilon, epsilon * epsilon_decay)

    return Q, returns, steps

def moving_average(x, k=10):
    x = np.asarray(x, dtype=float)
    if len(x) < k:
        return x
    w = np.ones(k) / k
    return np.convolve(x, w, mode="valid")

In [ ]:
# 训练 Q-learning（可直接运行这个单元格）
env = CliffWalkingEnv()
Q_ql, returns_ql, steps_ql = q_learning(
    env,
    episodes=600,
    alpha=0.5,
    gamma=0.95,
    epsilon=0.1,
    epsilon_decay=0.995,
    min_epsilon=0.01,
    seed=42
)

plt.figure()
plt.plot(returns_ql, label="Q-learning 每回合回报")
plt.xlabel("回合")
plt.ylabel("回报（越大越好）")
plt.legend()
plt.title("Q-learning 训练曲线")
plt.show()

print("Q-learning 学到的策略（箭头）示意：")
env.render_policy(Q_ql)

## 6. Dyna‑Q：把“学习到的模型”用于规划

- 在真实交互更新 Q 的同时，**拟合一个环境模型**：记录 $(s,a)\mapsto (r,s')$；
- 每步真实交互后，**从模型中采样**若干已见过的 $(s,a)$，进行 **规划更新（planning）**：
  $$
  Q(s,a)\gets Q(s,a) + \alpha\left[r + \gamma \max_{a'}Q(s',a') - Q(s,a)\right].
  $$
- 规划步数 `n_planning` 越多，**样本效率**通常越高，但计算量也增加。

**伪代码（简化）**：

```
Initialize Q(s,a), Model
for each episode:
    s ← reset()
    repeat:
        a ← ε-greedy(Q,s)
        s', r ← env.step(a)
        Q(s,a) ← Q(s,a) + α [r + γ max_{a'}Q(s',a') − Q(s,a)]
        Model(s,a) ← (r, s')

        repeat n_planning times:
            (ŝ, â) ← randomly sample from seen (s,a)
            (r̂, ŝ') ← Model(ŝ, â)
            Q(ŝ,â) ← Q(ŝ,â) + α [r̂ + γ max_{a'}Q(ŝ',a') − Q(ŝ,â)]
        s ← s'
    until done
```

In [ ]:
def dyna_q(env, episodes=300, alpha=0.5, gamma=0.95,
           epsilon=0.1, planning_steps=20, seed=0, max_steps_per_episode=1000):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.nS, env.nA))

    # 简单“模型”：用 dict 记忆 (s,a)->(r,s')
    model_r, model_s = {}, {}
    sa_seen = []  # 用于均匀采样

    returns, steps = [], []

    goal_state = env.goal[0] * env.width + env.goal[1]  # 终止状态 index

    for ep in range(episodes):
        s = env.reset()
        done = False
        G, t = 0.0, 0

        while not done and t < max_steps_per_episode:
            # 真实交互一步
            a = int(rng.integers(env.nA)) if rng.random() < epsilon else int(np.argmax(Q[s]))
            s2, r, done, _ = env.step(a)

            td_target = r + (0.0 if done else gamma * np.max(Q[s2]))
            Q[s, a] += alpha * (td_target - Q[s, a])

            # 更新模型
            model_r[(s, a)] = r
            model_s[(s, a)] = s2
            if (s, a) not in sa_seen:
                sa_seen.append((s, a))

            # 规划：从已见过的 (s,a) 随机采样并做同样的更新
            for _ in range(planning_steps):
                idx = int(rng.integers(len(sa_seen)))
                ss, aa = sa_seen[idx]
                rr = model_r[(ss, aa)]
                ss2 = model_s[(ss, aa)]
                # 终止判断：本环境只有到达 G 才终止
                target = rr + (0.0 if ss2 == goal_state else gamma * np.max(Q[ss2]))
                Q[ss, aa] += alpha * (target - Q[ss, aa])

            s = s2
            G += r
            t += 1

        returns.append(G)
        steps.append(t)

    return Q, returns, steps

In [ ]:
# 训练 Dyna-Q（可直接运行这个单元格）
env2 = CliffWalkingEnv()
Q_dyna, returns_dyna, steps_dyna = dyna_q(
    env2,
    episodes=300,
    alpha=0.5,
    gamma=0.95,
    epsilon=0.1,
    planning_steps=20,
    seed=42
)

# 可视化对比：每回合回报（移动平均以更平滑）
ma_ql   = moving_average(returns_ql, k=10)
ma_dyna = moving_average(returns_dyna, k=10)

plt.figure()
plt.plot(ma_ql,   label="Q-learning（10回合均值）")
plt.plot(ma_dyna, label="Dyna-Q（10回合均值）")
plt.xlabel("回合")
plt.ylabel("回报（越大越好）")
plt.title("Q-learning vs Dyna-Q（CliffWalking）")
plt.legend()
plt.show()

print("Dyna-Q 学到的策略（箭头）示意：")
env2.render_policy(Q_dyna)

## 8. 可视化与策略查看
- 上面两段训练代码会画出“每回合回报”的曲线；通常 **Dyna‑Q** 的收敛更快、样本效率更高；
- 你也可以调用 `env.render_policy(Q)` 在网格中查看最优策略箭头。

## 9. 调参与常见坑

**超参数建议（起点）**  
- 学习率 `α`：0.1～0.5；  
- 折扣 `γ`：0.95 或 0.99；  
- 探索 `ε`：0.1，或配合 `ε` 衰减 `ε_decay≈0.995`；  
- 规划步数 `n_planning`（Dyna‑Q）：5～50。

**常见坑**
- **悬崖环境终止条件**：踩悬崖并**不中止**，而是回到起点（容易写错）。
- **目标值的终止处理**：若 `done=True`，目标应去掉 $\gamma\max_{a'}Q(s',a')$。
- **状态编码**：小心二维坐标到一维 index 的转换。
- **曲线抖动**：用移动平均 `moving_average` 提升可读性。

## 10. 参考与延伸

- Sutton & Barto, *Reinforcement Learning: An Introduction*（第二版）
- “时序差分算法”：https://hrl.boyuai.com/chapter/1/时序差分算法  
- “Dyna‑Q算法”：https://hrl.boyuai.com/chapter/1/dyna-q算法

**可以继续做的改进**
- 把环境替换为你自己的任务（仍然能跑 Q‑learning / Dyna‑Q）。
- 加入更智能的模型（例如使用计数/平滑、或在随机环境里学习转移概率）。
- 尝试优先扫瞄（Prioritized Sweeping）等更高效的规划策略。